# Train GRPO Game

## Model & Tokenizer Loading

### Import

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

### Load Models

In [10]:
# We use Qwen3-0.6B-Instruct: small enough to train on a single GPU,
# but capable of following structured instructions.
MODEL_ID = "rd211/Qwen3-0.6B-Instruct"

# ── 1b. Load the tokenizer ────────────────────────────────────────────────────
# The tokenizer converts text ↔ token IDs.
# trust_remote_code=True is required for Qwen's custom tokenizer class.
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    padding_side="left",   # Left-padding is standard for causal LM generation
)
# Qwen3 uses a special end-of-sequence token; make sure pad = eos
tokenizer.pad_token = tokenizer.eos_token

# ── 1c. Load the base model in bfloat16 ──────────────────────────────────────
# bfloat16 halves memory vs float32 with minimal accuracy loss.
# device_map="auto" spreads layers across available GPUs/CPU automatically.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,   # Use bfloat16 for memory efficiency
    device_map="auto",            # Automatically place on GPU(s)
    trust_remote_code=True,
)

### Configure LoRA

In [11]:
# LoRA (Low-Rank Adaptation) adds small trainable matrices to certain layers.
# This means we only train ~1-2% of parameters instead of all 600M weights.
# r=16 is the rank — higher = more capacity but more memory.
# target_modules are the attention projection matrices inside the transformer.
lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=16,                          # LoRA rank (8–64 range typical)
    lora_alpha=32,                 # Scaling factor: effective lr = alpha/r * lr
    lora_dropout=0.05,             # Light dropout for regularization
    target_modules=[               # Apply LoRA to attention layers only
        "q_proj", "k_proj",
        "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",  # MLP layers too for Qwen3
    ],
    bias="none",                   # Don't train biases (saves memory)
)

# Wrap the model with LoRA adapters.
# Only the LoRA parameters will be updated during training.
model = get_peft_model(model, lora_config)

# Print a summary: should show ~1-3% trainable parameters
model.print_trainable_parameters()
# Expected output: "trainable params: 10,092,544 || all params: 606,142,464 || trainable%: 1.6650"

trainable params: 10,092,544 || all params: 606,142,464 || trainable%: 1.6650


## Game Environment

### Import

In [6]:
import random
from dataclasses import dataclass, field
from typing import Optional, Tuple

### LIAR'S DICE RULES

In [14]:
# Each player starts with 5 dice. Players take turns bidding on how many dice
# of a given face value (1–6) are on the table in total (both players' dice).
# A player can:
#   - BID: claim a higher quantity OR same quantity with higher face value
#   - CHALLENGE (call "Liar!"): if you think the last bid is false
# If challenged, all dice are revealed. If the bid is true → challenger loses a die.
# If bid is false → bidder loses a die. Game ends when one player has 0 dice.

@dataclass
class LiarsDiceState:
    """Represents a single game state (one player's turn)."""

    # The LLM-controlled player's dice (hidden from opponent)
    my_dice: list[int] = field(default_factory=list)

    # The opponent's dice count (we don't know their values)
    opponent_dice_count: int = 5

    # The current bid on the table: (quantity, face_value) or None for first move
    current_bid: Optional[Tuple[int, int]] = None

    # Whose turn it is: "llm" or "opponent"
    current_turn: str = "llm"

    # Track dice counts
    my_dice_count: int = 5


class LiarsDiceEnv:
    """
    Manages a Liar's Dice game and converts game states to text prompts
    that the LLM can understand and respond to.
    """

    def __init__(self, num_dice: int = 5):
        self.num_dice = num_dice
        self.reset()

    def reset(self) -> LiarsDiceState:
        """Start a new game: roll fresh dice for both players."""
        # Roll dice: values 1–6 for each die
        my_dice = [random.randint(1, 6) for _ in range(self.num_dice)]
        self.state = LiarsDiceState(
            my_dice=my_dice,
            opponent_dice_count=self.num_dice,
            current_bid=None,
            my_dice_count=self.num_dice,
        )
        return self.state

    def state_to_prompt(self, state: LiarsDiceState) -> str:
        """
        Convert a game state into a text prompt for the LLM.
        The quality of this prompt directly affects how well the model learns!
        We include:
          - The rules (for the model to reason about)
          - Current dice (the model's private info)
          - Current bid on table
          - Valid actions
        """
        # Describe dice as a readable summary
        dice_counts = {i: state.my_dice.count(i) for i in range(1, 7)}
        dice_desc = ", ".join(
            f"{count}x {face}" for face, count in dice_counts.items() if count > 0
        )

        # Describe the current bid
        if state.current_bid is None:
            bid_desc = "No bid yet — you go first."
            valid_actions = "You must make the opening bid."
        else:
            qty, face = state.current_bid
            bid_desc = f"Current bid: {qty} dice showing {face}s"
            valid_actions = (
                f"You can: (1) bid higher — either more than {qty} of any face, "
                f"or {qty}+ of a face > {face}, "
                f"(2) challenge — call the opponent a liar."
            )

        total_dice = state.my_dice_count + state.opponent_dice_count

        prompt = f"""You are playing Liar's Dice. Here is the current situation:

YOUR DICE ({state.my_dice_count} dice): {dice_desc}
OPPONENT'S DICE: {state.opponent_dice_count} (values hidden)
TOTAL DICE ON TABLE: {total_dice}
{bid_desc}

{valid_actions}

Think step by step about:
1. What dice do I have?
2. What is the probability the bid is true given all {total_dice} dice?
3. Should I bid higher or challenge?

Respond ONLY in this exact format:
<think>
[your reasoning here]
</think>
<action>
[BID quantity face_value] or [CHALLENGE]
Example: BID 3 4  (bidding 3 fours)
Example: CHALLENGE
</action>"""
        return prompt

    def parse_action(self, response: str) -> Optional[Tuple[str, Optional[Tuple[int, int]]]]:
        """
        Parse the LLM's text response into a structured action.
        Returns: ("BID", (quantity, face)) or ("CHALLENGE", None) or ("INVALID", None)
        """
        import re

        # Extract content inside <action> tags
        action_match = re.search(r"<action>\s*(.*?)\s*</action>", response, re.DOTALL)
        if not action_match:
            return ("INVALID", None)  # No action tag found

        action_text = action_match.group(1).strip().upper()

        if action_text == "CHALLENGE":
            return ("CHALLENGE", None)

        # Try to parse BID quantity face_value
        bid_match = re.match(r"BID\s+(\d+)\s+(\d+)", action_text)
        if bid_match:
            qty = int(bid_match.group(1))
            face = int(bid_match.group(2))
            if 1 <= face <= 6 and qty >= 1:  # Basic validity
                return ("BID", (qty, face))

        return ("INVALID", None)  # Could not parse

    def is_valid_action(
        self,
        action: Tuple[str, Optional[Tuple[int, int]]],
        state: LiarsDiceState
    ) -> bool:
        """
        Check if a parsed action is valid given the current game state.
        This is the core of our verifiable reward!
        """
        action_type, bid_data = action

        if action_type == "INVALID":
            return False  # Malformed response

        if action_type == "CHALLENGE":
            # You can always challenge if there's an existing bid
            return state.current_bid is not None

        if action_type == "BID":
            qty, face = bid_data
            if state.current_bid is None:
                # Opening bid: anything goes (qty ≥ 1, face 1–6)
                return True
            prev_qty, prev_face = state.current_bid
            # Valid bid: higher quantity, OR same quantity with higher face
            return (qty > prev_qty) or (qty == prev_qty and face > prev_face)

        return False

    def evaluate_challenge(self, state: LiarsDiceState, opponent_dice: list[int]) -> bool:
        """
        Resolve a CHALLENGE. Returns True if the challenger (LLM) wins.
        In a real game, both players reveal dice.
        For training, we simulate the opponent's dice.
        """
        if state.current_bid is None:
            return True  # Can't challenge with no bid

        qty, face = state.current_bid
        total_dice = state.my_dice + opponent_dice
        actual_count = total_dice.count(face)

        # Challenger wins if the actual count is LESS than the bid
        return actual_count < qty

## Prompt Dataset Generation

### Import

In [12]:
import random
from datasets import Dataset

### Generate Dataset

In [15]:
def generate_liars_dice_prompts(num_samples: int = 2000) -> Dataset:
    """
    Generate a diverse dataset of Liar's Dice game situations.

    GRPO doesn't need labeled answers — it only needs PROMPTS.
    The model generates its own responses, and we score them with reward functions.
    
    We want diversity:
      - Early game (lots of dice, no bid yet)
      - Mid game (ongoing bids)
      - Late game (few dice, bluffing situations)
    """
    env = LiarsDiceEnv()
    prompts = []

    for i in range(num_samples):
        # Randomly vary the game state for diversity
        my_dice_count = random.randint(1, 5)
        opp_dice_count = random.randint(1, 5)

        # Roll random dice
        my_dice = [random.randint(1, 6) for _ in range(my_dice_count)]

        # Randomly decide if there's an existing bid
        if random.random() < 0.7:
            # Mid-game: there's already a bid
            total = my_dice_count + opp_dice_count
            # Generate a plausible (but not always optimal) bid
            bid_qty = random.randint(1, max(1, total // 2 + 1))
            bid_face = random.randint(1, 6)
            current_bid = (bid_qty, bid_face)
        else:
            # Opening move
            current_bid = None

        state = LiarsDiceState(
            my_dice=my_dice,
            opponent_dice_count=opp_dice_count,
            current_bid=current_bid,
            my_dice_count=my_dice_count,
        )

        # Convert state to the chat format GRPOTrainer expects:
        # A list of message dicts (like a chat conversation)
        prompt_text = env.state_to_prompt(state)

        # Store both the formatted prompt AND the raw state for reward computation
        prompts.append({
            "prompt": [
                {"role": "system", "content": "You are an expert Liar's Dice player."},
                {"role": "user",   "content": prompt_text},
            ],
            # Store metadata as strings (HuggingFace Dataset requires serializable types)
            "my_dice": str(my_dice),
            "current_bid": str(current_bid),
            "my_dice_count": my_dice_count,
            "opp_dice_count": opp_dice_count,
        })

    # Shuffle for training variety
    random.shuffle(prompts)

    # Convert to HuggingFace Dataset format
    dataset = Dataset.from_list(prompts)

    # Split into train/eval
    dataset = dataset.train_test_split(test_size=0.05, seed=42)
    return dataset


# Generate the dataset
print("Generating game prompts...")
dataset = generate_liars_dice_prompts(num_samples=3000)
print(f"Train: {len(dataset['train'])} | Eval: {len(dataset['test'])}")
# Save to disk so we don't regenerate every run
dataset.save_to_disk("liars_dice_dataset")

Generating game prompts...
Train: 2850 | Eval: 150


Saving the dataset (0/1 shards):   0%|          | 0/2850 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/150 [00:00<?, ? examples/s]

## Reward Functions

### Import

In [16]:
import re
import ast
import random
from typing import Optional

### Normalize Completions

In [69]:
def normalize_completions(completions: list) -> list[str]:
    if not completions:
        return []

    # Case 1: List[List[Dict]]
    if isinstance(completions[0][0], dict):
        return [completion[0].get("content", "") for completion in completions]

    # Case 2: List[List[str]]
    elif isinstance(completions[0][0], str):
        return [completion[0] for completion in completions]

    # Case 3: List[str]
    elif isinstance(completions[0], str):
        return completions

    else:
        raise TypeError(
            "Unsupported format for completions. Expected List[List[Dict]] or List[List[str]]."
        )

### Format Reward

In [70]:
# Reward the model for following the required output format.
# This is crucial: if the format is wrong, we can't parse the action at all.
# Start with format rewards so the model first learns HOW to respond.

def format_reward(completions: list[str], **kwargs) -> list[float]:
    """
    Returns +0.5 if the response has both <think> and <action> tags,
    +0.25 for just one tag, 0.0 if neither.
    
    completions: list of model-generated strings (one per sample in the batch)
    **kwargs: additional columns from the dataset row (my_dice, current_bid, etc.)
    """
    completions = normalize_completions(completions)
    rewards = []
    for completion in completions:
        has_think  = bool(re.search(r"<think>.*?</think>",   completion, re.DOTALL))
        has_action = bool(re.search(r"<action>.*?</action>", completion, re.DOTALL))

        if has_think and has_action:
            rewards.append(0.5)   # Full format reward
        elif has_think or has_action:
            rewards.append(0.25)  # Partial credit
        else:
            rewards.append(0.0)   # No format at all
    return rewards

### Valid Move Reward

In [71]:
# The core "game rules" reward. Did the model make a legal move?
# This is a VERIFIABLE reward — no ambiguity, no need for human judges.

def valid_move_reward(completions: list[str], **kwargs) -> list[float]:
    """
    Returns +1.0 for a valid game action, 0.0 for invalid.
    
    The kwargs will contain columns from our dataset:
      - current_bid: string representation of (qty, face) tuple or "None"
      - my_dice_count: int
    """
    rewards = []
    env = LiarsDiceEnv()
    
    completions = normalize_completions(completions)
    for i, completion in enumerate(completions):
        # Reconstruct the game state from the stored metadata
        current_bid_str = kwargs.get("current_bid", ["None"] * len(completions))[i]
        my_dice_count   = kwargs.get("my_dice_count", [5] * len(completions))[i]
        opp_dice_count  = kwargs.get("opp_dice_count", [5] * len(completions))[i]

        # Parse the stored bid back from string
        try:
            current_bid = ast.literal_eval(current_bid_str)  # "(3, 4)" → (3, 4)
        except (ValueError, SyntaxError):
            current_bid = None

        state = LiarsDiceState(
            my_dice=[],       # Not needed for move validation
            current_bid=current_bid,
            my_dice_count=int(my_dice_count),
            opponent_dice_count=int(opp_dice_count),
        )

        # Parse the LLM's action
        action = env.parse_action(completion)

        # Check if it's a valid game action
        if env.is_valid_action(action, state):
            rewards.append(1.0)   # Legal move!
        else:
            rewards.append(0.0)   # Illegal move or malformed response

    return rewards

### Strategy Reward

In [72]:
# Reward good strategic thinking. This is more subjective but still rule-based.
# 
# Strategy heuristics for Liar's Dice:
#   - If you have many of a face value, bidding that face is smart
#   - Challenging a bid that seems impossible given your dice is smart
#   - Bidding extremely high quantities is risky (penalize)

def strategy_reward(completions: list[str], **kwargs) -> list[float]:
    """
    Returns a float in [-0.5, +0.5] based on strategic quality.
    Positive = strategically sound, negative = poor strategy.
    """
    rewards = []
    env = LiarsDiceEnv()
    
    completions = normalize_completions(completions)
    for i, completion in enumerate(completions):
        # Get dice information
        my_dice_str   = kwargs.get("my_dice", ["[]"]   * len(completions))[i]
        current_bid_str = kwargs.get("current_bid", ["None"] * len(completions))[i]
        my_dice_count = int(kwargs.get("my_dice_count", [5] * len(completions))[i])
        opp_dice_count = int(kwargs.get("opp_dice_count", [5] * len(completions))[i])

        try:
            my_dice     = ast.literal_eval(my_dice_str)
            current_bid = ast.literal_eval(current_bid_str)
        except (ValueError, SyntaxError):
            rewards.append(0.0)
            continue

        action = env.parse_action(completion)
        action_type, bid_data = action

        if action_type == "INVALID":
            rewards.append(-0.5)   # Penalize malformed responses
            continue

        total_dice = my_dice_count + opp_dice_count
        reward = 0.0

        if action_type == "BID" and bid_data:
            qty, face = bid_data
            # How many of this face do I personally have?
            my_count_of_face = my_dice.count(face) if my_dice else 0

            # Heuristic: good bid if my dice support it
            # Expected count in total = (total_dice * 1/6) + my_dice bias
            expected_total = total_dice / 6 + my_count_of_face * 0.5
            if qty <= expected_total:
                reward += 0.3   # Conservative, statistically sound bid
            elif qty <= expected_total * 1.5:
                reward += 0.1   # Slightly risky bluff
            else:
                reward -= 0.3   # Very risky overbid

        elif action_type == "CHALLENGE" and current_bid:
            qty, face = current_bid
            my_count_of_face = my_dice.count(face) if my_dice else 0
            # Expected count if evenly distributed
            expected_total = total_dice / 6 + my_count_of_face * 0.5
            if qty > expected_total * 1.3:
                reward += 0.5   # Smart challenge — bid seems too high
            elif qty < expected_total * 0.7:
                reward -= 0.3   # Unwise challenge — bid seems low/reasonable

        rewards.append(max(-0.5, min(0.5, reward)))  # Clamp to [-0.5, 0.5]

    return rewards

### Reasoning Quality Reward

In [73]:
# Reward the model for including substantive reasoning in <think> tags.
# This encourages chain-of-thought behavior (like DeepSeek-R1).

def reasoning_reward(completions: list[str], **kwargs) -> list[float]:
    """
    Returns +0.3 if <think> block mentions relevant concepts (dice, probability, etc.)
    Returns 0.0 if <think> is present but empty/trivial
    Returns -0.1 if no <think> tag at all
    """
    rewards = []
    # Keywords we want to see in the reasoning
    reasoning_keywords = [
        "dice", "probability", "chance", "bid", "challenge",
        "bluff", "risk", "count", "face", "likely", "unlikely",
    ]
    
    completions = normalize_completions(completions)
    for completion in completions:
        think_match = re.search(r"<think>(.*?)</think>", completion, re.DOTALL)

        if not think_match:
            rewards.append(-0.1)   # No reasoning at all
            continue

        think_text = think_match.group(1).lower()

        # Count how many reasoning keywords appear
        keyword_hits = sum(1 for kw in reasoning_keywords if kw in think_text)

        if len(think_text.strip()) < 20:
            rewards.append(0.0)    # Reasoning exists but is trivially short
        elif keyword_hits >= 3:
            rewards.append(0.3)    # Good, substantive reasoning
        elif keyword_hits >= 1:
            rewards.append(0.15)   # Some relevant reasoning
        else:
            rewards.append(0.05)   # Reasoning present but not very relevant

    return rewards

## GRPO Training Config & Trainer

### Import

In [74]:
from datasets import load_from_disk
from trl import GRPOConfig, GRPOTrainer

### Load Dataset

In [75]:
dataset = load_from_disk("liars_dice_dataset")
train_dataset = dataset["train"]
eval_dataset  = dataset["test"]

### Training Configuration

In [76]:
training_args = GRPOConfig(
    # ── Output ────────────────────────────────────────────────────────────────
    output_dir="qwen3-0.6b-liars-dice-grpo",

    # ── Core GRPO parameters ──────────────────────────────────────────────────
    # num_generations: how many responses to sample per prompt
    # More = better advantage estimates but more memory. Start with 4, try 8.
    num_generations=8,

    # max_completion_length: max tokens the model generates per response
    # Enough for <think>reasoning</think><action>BID 3 4</action>
    max_completion_length=256,

    # max_prompt_length: max tokens for the game state prompt
    max_prompt_length=512,

    # ── Optimization ──────────────────────────────────────────────────────────
    learning_rate=5e-6,               # Low LR for stable RL fine-tuning
    per_device_train_batch_size=2,     # Prompts per device per step (not completions!)
    gradient_accumulation_steps=4,    # Effective batch = 2 * 4 = 8 prompts
    num_train_epochs=3,                # 3 epochs over the dataset

    # ── Mixed precision ────────────────────────────────────────────────────────
    bf16=True,    # Use bfloat16 (requires Ampere GPU or newer: RTX 30xx+, A100)
    fp16=False,   # Don't use fp16 alongside bf16

    # ── Logging and saving ────────────────────────────────────────────────────
    logging_steps=10,
    save_steps=100,
    eval_steps=50,
    report_to="wandb",        # Change to "none" if not using wandb
    run_name="liars-dice-grpo-v1"
)

### Initialize the GRPO Trainer

In [77]:
# Pass ALL reward functions as a list — TRL sums them automatically.
# You can also weight them: reward_weights=[1.0, 2.0, 0.5, 0.3]
trainer = GRPOTrainer(
    model=model,                       # The LoRA-wrapped Qwen3 model
    processing_class=tokenizer,        # Tokenizer for encoding/decoding
    args=training_args,

    # Reward functions: each takes (completions, **kwargs) and returns list[float]
    reward_funcs=[
        format_reward,      # Weight: 1.0 (format compliance)
        valid_move_reward,  # Weight: 1.0 (game rule validity)
        strategy_reward,    # Weight: 1.0 (strategic quality)
        reasoning_reward,   # Weight: 1.0 (chain-of-thought quality)
    ],

    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


## Running & Monitoring Training

### Import

In [40]:
import wandb

### Initialize wandb

In [30]:
# wandb gives you live plots of reward, loss, KL divergence, etc.
# If you don't have wandb, set report_to="none" in GRPOConfig above.
wandb.init(
    project="grpo-game-ai",
    name="liars-dice-qwen3-0.6b",
    config={
        "model": MODEL_ID,
        "game": "liars_dice",
        "num_generations": 8,
        "lora_r": 16,
    }
)

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: gondar (gondar-team) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


### Start training

In [ ]:
# This is the main training loop. TRL handles:
#   1. Generating N completions per prompt
#   2. Calling your reward functions
#   3. Computing advantages via Z-score normalization
#   4. Computing GRPO loss (clipped policy ratio + KL penalty)
#   5. Backpropagating through the LoRA layers
#   6. Logging metrics

print("Starting GRPO training...")
train_result = trainer.train()

Starting GRPO training...


Step,Training Loss
10,0.000000
20,0.000000
30,0.000000
40,0.008100
50,0.000000
60,0.000800
70,0.000000
80,0.000000
90,0.000000
100,0.000000


wandb: WARNING The get_url method is deprecated and will be removed in a future release. Please use `run.url` instead.


### Save the trained LoRA Adapter

In [ ]:
# This saves only the small LoRA delta weights, not the full model.
trainer.save_model("qwen3-0.6b-liars-dice-grpo/final")
tokenizer.save_pretrained("qwen3-0.6b-liars-dice-grpo/final")

print("Training complete! Model saved.")
print(f"Final metrics: {train_result.metrics}")

## Evaluate & Testing

### Import

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

### Load the trained model

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-0.6B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
# Load LoRA adapter on top of the base model
model = PeftModel.from_pretrained(base_model, "qwen3-0.6b-liars-dice-grpo/final")
model = model.merge_and_unload()   # Merge LoRA weights into base for faster inference

tokenizer = AutoTokenizer.from_pretrained("qwen3-0.6b-liars-dice-grpo/final")

### Quick evaluation: % of valid moves over N random states

In [ ]:
def play_one_turn(state, verbose=True):
    """Run the trained model on a game state and return its action."""
    env = LiarsDiceEnv()
    prompt_text = env.state_to_prompt(state)

    # Format as a chat message
    messages = [
        {"role": "system",  "content": "You are an expert Liar's Dice player."},
        {"role": "user",    "content": prompt_text},
    ]

    # Apply the chat template (Qwen3 uses a specific format)
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,       # Some randomness for varied play
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],   # Only decode newly generated tokens
        skip_special_tokens=True,
    )

    if verbose:
        print("=== Model Response ===")
        print(response)
        print("=== Parsed Action ===")
        action = env.parse_action(response)
        print(f"Action: {action}")
        valid = env.is_valid_action(action, state)
        print(f"Valid: {valid}")

    return response

def evaluate_validity(num_trials: int = 100) -> dict:
    """
    Run the model on N random game states and measure:
      - % of valid moves (primary metric)
      - % with correct format
      - Action type distribution (bid vs challenge)
    """
    import random
    env = LiarsDiceEnv()

    valid_count = 0
    format_count = 0
    action_types = {"BID": 0, "CHALLENGE": 0, "INVALID": 0}

    for _ in range(num_trials):
        env.reset()
        # Random mid-game state
        state = env.state
        state.current_bid = (random.randint(1, 3), random.randint(1, 6)) if random.random() > 0.3 else None

        response = play_one_turn(state, verbose=False)
        action = env.parse_action(response)
        action_type = action[0]
        action_types[action_type] += 1

        if env.is_valid_action(action, state):
            valid_count += 1

        import re
        if re.search(r"<think>.*?</think>.*<action>.*?</action>", response, re.DOTALL):
            format_count += 1

    return {
        "valid_move_rate": valid_count / num_trials,
        "format_compliance": format_count / num_trials,
        "action_distribution": {k: v/num_trials for k, v in action_types.items()},
    }

### Run Evaluation

In [ ]:
print("\n=== Evaluating trained model ===")
results = evaluate_validity(100)
print(f"Valid move rate:    {results['valid_move_rate']:.1%}")
print(f"Format compliance: {results['format_compliance']:.1%}")
print(f"Action mix:        {results['action_distribution']}")